# Tutorial to use ImputeGAP

## 1. Creating and visualizing an existent dataset

### 1.1. Create a timeseries object, load a series, normalize it

If I correctly understood the .txt series is a 2d representation of the dataset, each column is a channel, each row is a timestep, and then if you have multiple samples, you simply concatenate them as new rows. Then it is up to me to choose nbr_val to cut the samples.

In this case eeg-alcohol is a an individual sample, with 64 channels (features) and 256 values (time steps in a way).

In [1]:
from imputegap.recovery.manager import TimeSeries
from imputegap.tools import utils

# initialize the time series object
ts = TimeSeries()

# load and normalize the dataset from the library
ts.load_series(utils.search_path("eeg-alcohol"))
ts.normalize(normalizer="z_score")

# print and plot a subset of time series
ts.print(nbr_series=6, nbr_val=20)
ts.plot(input_data=ts.data, nbr_series=6, nbr_val=100, save_path="./imputegap_assets")


(SYS) The dataset is loaded from /mnt/fast/projects/ds_sem_repos/imputegap_repo/ImputeGAP/imputegap/datasets/eeg-alcohol.txt

> logs: normalization (z_score) of the data - runtime: 0.0005 seconds

shape of eeg-alcohol : (64, 256)
	number of series = 64
	number of values = 256

                  TS_1           TS_2           TS_3           TS_4           TS_5           TS_6           
idx_0             2.1903285853   1.6705262632   2.2587543576   2.2311576144   2.0642608690   2.0352202783
idx_1             1.9472603285   1.5655748037   2.1966398110   2.2311576144   1.9054970599   1.9505308899
idx_2             1.7041920717   1.4605158119   2.1966398110   2.3667495605   1.7468957521   1.7814984911
idx_3             1.5825957138   1.3029810904   2.1345252645   2.3667495605   1.5088312891   1.6122929032
idx_4             1.4003878654   1.0929706391   1.8858125103   2.1633616414   1.3500674799   1.3585711159
idx_5             1.2180555580   0.7779011960   1.5751124936   1.7563079509   1.19

'./imputegap_assets/25_11_28_13_53_33_imputegap_plot.jpg'

In [2]:
# (channels, values)
ts.data.shape

(64, 256)

### 1.2. List of all the datasets available

In [3]:
from imputegap.recovery.manager import TimeSeries
ts = TimeSeries()
print(f"ImputeGAP datasets : {ts.datasets}")

ImputeGAP datasets : ['airq', 'bafu', 'chlorine', 'climate', 'drift', 'eeg-alcohol', 'eeg-reading', 'electricity', 'fmri-stoptask', 'forecast-economy', 'meteo', 'motion', 'soccer', 'solar-plant', 'sport-activity', 'stock-exchange', 'temperature', 'traffic']


## 2. Working with Physionet

Physionet dataset has 35 channels (physiological parameters measure in the ICU), and 48 values (measurements taken for 2 days each hour).

In [4]:
import numpy as np

npz = np.load("external/physionet/physionet.npz")
print(npz.files)

['x_train_full', 'x_train_miss', 'm_train_miss', 'm_train_artificial', 'y_train', 'x_val_full', 'x_val_miss', 'm_val_miss', 'm_val_artificial', 'y_val', 'x_test_full', 'x_test_miss', 'm_test_miss', 'm_test_artificial', 'y_test']


In [5]:
full_data = npz["x_train_full"]
print(npz['y_train'].shape)
print(npz['y_val'].shape)
print(npz['y_test'].shape)
print(full_data.shape)

(3997,)
(3993,)
(3997,)
(3997, 48, 35)


### 2.1. Using import_matrix, the method expects (Series, Values)

For the case of physionet it is necessary to do a transpose in order to adjust the data dimensions to agree with (series, values).

In [6]:
# select patient with index 27 in the (Patients, Values, Series) array
one_patient = full_data[27]
# transpose the patient data (Values, Series) to get (Series, Values)
one_patient_compatible = one_patient.transpose()

In [7]:
# full_data shape (Patients, Values, Series)
# what we want is (Series, Values*Patients)
# First transpose (P, V, S) to get (S, P, V), then reshape to get (S, P*V) (with patients values
# concatenated)
full_data_compatible = full_data.transpose(2,0,1).reshape(35,-1)

#### 2.1.1. Test that the conversion is correct

In [8]:
sample_idx = 27
values_one_sample = 48
# Idx to access the first value for a patient with a given sample idx (flattened dimension)
flatten_idx = sample_idx*values_one_sample
patient_test = full_data_compatible[:, flatten_idx:flatten_idx+values_one_sample]

In [9]:
np.all(one_patient_compatible == patient_test)

True

#### 2.1.2. Produce the same plot from the single patient and with full data

Nomrmalization consists in normalizing the values in each channel, not sure whether it depends only on the channel or also inter-channel dependency for normalization. In any case, if you multiple patients data or a single one, the two plots for a single patient will change since the normalization is different.

In [10]:
# import data as the matrix with one single patient
from imputegap.recovery.manager import TimeSeries
ts = TimeSeries()
ts.import_matrix(one_patient_compatible)
# ts.normalize(normalizer="z_score")
ts.print()
ts.plot(input_data=ts.data, nbr_series=20, nbr_val=48, save_path="./imputegap_assets")


shape of default : (35, 48)
	number of series = 35
	number of values = 48

                  TS_1           TS_2           TS_3           TS_4           TS_5           TS_6           TS_7           
idx_0             0.1121087000   1.1960027218   0.0000000000   0.0000000000  -0.5403673649   0.0000000000   0.0000000000
idx_1            -0.8099491596  -0.0968640596   0.0000000000   0.0000000000   0.1236104965   0.0000000000   0.0000000000
idx_2             0.0000000000   1.2522143126  -0.7848144174   0.0000000000  -1.8683230877   0.0000000000   0.0000000000
idx_3             0.0000000000   0.1279823333   0.0000000000   0.0000000000   1.2523728609   0.0000000000   0.0000000000
idx_4             0.0000000000   0.1279823333   0.0000000000   0.0000000000  -0.9387540817   0.0000000000  -0.0353275612
idx_5             0.0000000000   0.1279823333   0.0000000000   0.0000000000  -1.7355275154   0.0000000000   0.0000000000
idx_6             0.0000000000   0.6900982857   0.0000000000   0.000000000

'./imputegap_assets/25_11_28_13_54_10_imputegap_plot.jpg'

In [11]:
# import data as the matrix with all the patient values concatenated
from imputegap.recovery.manager import TimeSeries
ts = TimeSeries()
ts.import_matrix(full_data_compatible)
# ts.normalize(normalizer="z_score")
ts.print()
# to select a specific patient, a slice the input data manually, there doesn't seem
# to be a param to do so in plot.
ts.plot(input_data=ts.data[:,flatten_idx:flatten_idx+48], nbr_series=20, nbr_val=48, save_path="./imputegap_assets")


shape of default : (35, 191856)
	number of series = 35
	number of values = 191856

                  TS_1           TS_2           TS_3           TS_4           TS_5           TS_6           TS_7           
idx_0             0.0000000000   0.4090403318   0.0000000000   0.0000000000   2.0491464138   0.9948062301   1.3635373116
idx_1             0.0000000000   0.0717707351   0.0000000000   0.0000000000   1.0531795025   0.0000000000   0.0000000000
idx_2             0.0000000000  -0.2092872560   0.0000000000   0.0000000000  -0.0755828619   0.0000000000   0.0000000000
idx_3             0.0000000000   0.0717707351   0.0000000000   0.0000000000   0.1900082827   0.0000000000   0.0000000000
idx_4             0.0000000000   0.7463099360   0.0000000000   0.0000000000   1.7835551500   0.0000000000   0.0000000000
idx_5             0.0000000000   0.8587331176   0.0000000000   0.0000000000   1.5179640055   0.0000000000   0.0000000000
idx_6             0.0000000000   1.0835795403  -0.3991306126   0.0

'./imputegap_assets/25_11_28_13_54_14_imputegap_plot.jpg'

### 2.2. Create a 2d .txt file for a 3d datasest

In [12]:
import numpy as np
import os

npz = np.load("external/physionet/physionet.npz")

out_dir = "external/datasets"
os.makedirs(out_dir, exist_ok=True)

def convert(split):
    X = npz[f"x_{split}_full"]  # shape (P, V, S)
    P, V, S = X.shape
    X_flat = X.transpose(2,0,1).reshape(S,-1).transpose()

    out_path = os.path.join(out_dir, f"{split}.txt")
    np.savetxt(out_path, X_flat, fmt="%.6f")
    print(f"{split}: from {X.shape} to {X_flat.shape}  ->  saved to {out_path}")

convert("train")
convert("val")
convert("test")

train: from (3997, 48, 35) to (191856, 35)  ->  saved to external/datasets/train.txt
val: from (3993, 48, 35) to (191664, 35)  ->  saved to external/datasets/val.txt
test: from (3997, 48, 35) to (191856, 35)  ->  saved to external/datasets/test.txt


#### 2.2.1. Verify that the load is correct

In [13]:
from imputegap.recovery.manager import TimeSeries
ts = TimeSeries()
ts.load_series("./external/datasets/train.txt")
ts.print()
# do the plot again for the patient with id 27
ts.plot(input_data=ts.data[:,flatten_idx:flatten_idx+48], nbr_series=20, nbr_val=48, save_path="./imputegap_assets")


(SYS) The dataset is loaded from ./external/datasets/train.txt


shape of default : (35, 191856)
	number of series = 35
	number of values = 191856

                  TS_1           TS_2           TS_3           TS_4           TS_5           TS_6           TS_7           
idx_0             0.0000000000   0.4090400000   0.0000000000   0.0000000000   2.0491460000   0.9948060000   1.3635370000
idx_1             0.0000000000   0.0717710000   0.0000000000   0.0000000000   1.0531800000   0.0000000000   0.0000000000
idx_2             0.0000000000  -0.2092870000   0.0000000000   0.0000000000  -0.0755830000   0.0000000000   0.0000000000
idx_3             0.0000000000   0.0717710000   0.0000000000   0.0000000000   0.1900080000   0.0000000000   0.0000000000
idx_4             0.0000000000   0.7463100000   0.0000000000   0.0000000000   1.7835550000   0.0000000000   0.0000000000
idx_5             0.0000000000   0.8587330000   0.0000000000   0.0000000000   1.5179640000   0.0000000000   0.0000000000
i

'./imputegap_assets/25_11_28_13_54_55_imputegap_plot.jpg'

### 2.3. Data contamination

In [ ]:
from imputegap.recovery.manager import TimeSeries

ts = TimeSeries()
ts.load_series("./external/datasets/train.txt")
ts.normalize(normalizer="z_score")
ts.print()

# contaminate the time series with MCAR pattern (patient 27)
# rate_dataset = how many channels have something missing
# rate_series = percentage of data missing in the channel
ts_m = ts.Contamination.mcar(ts.data[:,flatten_idx:flatten_idx+48], rate_dataset=0.8, rate_series=0.4, block_size=10, seed=True)

# do the plot again for the patient with id 27
ts.plot(input_data=ts.data[:,flatten_idx:flatten_idx+48], incomp_data=ts_m, nbr_series=20, nbr_val=48, subplot=True, save_path="./imputegap_assets")


(SYS) The dataset is loaded from ./external/datasets/train.txt

> logs: normalization (z_score) of the data - runtime: 0.0740 seconds

shape of default : (35, 191856)
	number of series = 35
	number of values = 191856

                  TS_1           TS_2           TS_3           TS_4           TS_5           TS_6           TS_7           
idx_0             0.0000000019   0.4309007871   0.0000000122   0.0000000024   3.1583463059   2.9115963263   5.2688999868
idx_1             0.0000000019   0.0756067048   0.0000000122   0.0000000024   1.6232650916   0.0000000031   0.0000000050
idx_2             0.0000000019  -0.2204722238   0.0000000122   0.0000000024  -0.1164959813   0.0000000031   0.0000000050
idx_3             0.0000000019   0.0756067048   0.0000000122   0.0000000024   0.2928591120   0.0000000031   0.0000000050
idx_4             0.0000000019   0.7861959229   0.0000000122   0.0000000024   2.7489912127   0.0000000031   0.0000000050
idx_5             0.0000000019   0.9046272837   0.00

'./imputegap_assets/25_11_28_14_05_22_imputegap_plot.jpg'

In [21]:
from imputegap.recovery.manager import TimeSeries
ts = TimeSeries()
print(f"Missingness patterns : {ts.patterns}")

Missingness patterns : ['aligned', 'disjoint', 'distribution', 'gaussian', 'mcar', 'overlap', 'scattered']


### 2.4. Imputation

I had to install this on my ubuntu machine:
```bash
sudo apt-get install libopenblas0 libopenblas-dev
```

In [ ]:
from imputegap.recovery.imputation import Imputation
from imputegap.recovery.manager import TimeSeries
from imputegap.tools import utils

# initialize the time series object
ts = TimeSeries()

# load and normalize the dataset
ts.load_series("./external/datasets/train.txt")
ts.normalize(normalizer="z_score")

# impute the contaminated series
# using ts_m generated in the previous cell for patient 27
imputer = Imputation.MatrixCompletion.CDRec(ts_m)
imputer.impute()

# compute and print the imputation metrics
imputer.score(ts.data[:,flatten_idx:flatten_idx+48], imputer.recov_data)
ts.print_results(imputer.metrics)

# plot the recovered time series
ts.plot(input_data=ts.data[:,flatten_idx:flatten_idx+48], incomp_data=ts_m, recov_data=imputer.recov_data, nbr_series=9, subplot=True, algorithm=imputer.algorithm, save_path="./imputegap_assets/imputation")


(SYS) The dataset is loaded from ./external/datasets/train.txt

> logs: normalization (z_score) of the data - runtime: 0.0713 seconds

(IMPUTATION) CDRec
	Matrix: 35, 48
	truncation rank: 3
	epsilon: 1e-06
	iterations: 100

> logs: imputation cdrec - Execution Time: 0.0013 seconds.

Results :
RMSE                 = 0.6451849114146669
MAE                  = 0.2238358134370778
MI                   = 0.29289508423722826
CORRELATION          = 0.5096296400470466

plots saved in: ./imputegap_assets/imputation/25_11_28_14_30_07_cdrec_plot.jpg


'./imputegap_assets/imputation/25_11_28_14_30_07_cdrec_plot.jpg'

In [ ]:
config = {"rank": 5, "epsilon": 0.01, "iterations": 100}
imputer.impute(params=config)

In [25]:
from imputegap.recovery.manager import TimeSeries
ts = TimeSeries()
print(f"Imputation families : {ts.families}")
print(f"Imputation algorithms : {ts.algorithms}")

Imputation families : ['DeepLearning', 'LLMs', 'MachineLearning', 'MatrixCompletion', 'PatternSearch', 'Statistics']
Imputation algorithms : ['BRITS', 'BayOTIDE', 'BitGraph', 'CDRec', 'DeepMVI', 'DynaMMo', 'GAIN', 'GPT4TS', 'GRIN', 'GROUSE', 'HKMF_T', 'IIM', 'Interpolation', 'IterativeSVD', 'KNNImpute', 'MICE', 'MPIN', 'MRNN', 'MeanImpute', 'MeanImputeBySeries', 'MinImpute', 'MissForest', 'MissNet', 'NuwaTS', 'PRISTI', 'ROSL', 'SPIRIT', 'STMVL', 'SVT', 'SoftImpute', 'TKCM', 'TRMF', 'XGBOOST', 'ZeroImpute']


### 2.5. Others (still to check)

#### Parameter Tuning

In [26]:
from imputegap.recovery.manager import TimeSeries
ts = TimeSeries()
print(f"AutoML Optimizers : {ts.optimizers}")

AutoML Optimizers : ['bayesian', 'greedy', 'particle_swarm', 'ray_tune', 'successive_halving']


#### Benchmark

In [ ]:
from imputegap.recovery.benchmark import Benchmark

my_algorithms = ["SoftImpute", "MeanImpute"]

my_opt = ["default_params"]

my_datasets = ["eeg-alcohol"]

my_patterns = ["mcar"]

range = [0.05, 0.1, 0.2, 0.4, 0.6, 0.8]

my_metrics = ["*"]

# launch the evaluation
bench = Benchmark()
bench.eval(algorithms=my_algorithms, datasets=my_datasets, patterns=my_patterns, x_axis=range, metrics=my_metrics, optimizers=my_opt)

## 3. Use trained model on Physionet

In [ ]:
# ============================
# Flags for defaults
# ============================

K = 1                                 # Number of importance sampling weights
M = 1                                 # Number of samples for ELBO estimation

# banded_covar = False                  # Use banded covariance (only for gp-vae)

# basedir = "models"                    # Directory for saving models

# batch_size = 64                       # Training batch size

# cnn_kernel_size = 3                   # Kernel size of CNN preprocessor
# cnn_sizes = [256]                     # Number of filters per CNN layer (comma-separated list → list)

# data_dir = ""                         # Directory where data is read from
# data_type = "hmnist"                  # <hmnist | physionet | sprites>

# exp_name = "debug"                    # Experiment name

# gradient_clip = 10000.0               # Max global gradient norm

kernel = "cauchy"                     # GP kernel <rbf|diffusion|matern|cauchy>
kernel_scales = 1                     # Number of GP kernel length scales

# learning_rate = 0.001                 # LR

# model_type = "gp-vae"                 # <vae | hi-vae | gp-vae>

# num_steps = 0                         # Number of training steps (0 → use num_epochs)
# print_interval = 0                    # Interval for printing/saving

# seed = 1337                           # RNG seed

# testing = False                       # Whether to use the test set

#=============================
# Specific flags for Physionet experiment
#=============================
latent_dim = 35                    # override
encoder_sizes = [128, 128]         # override
decoder_sizes = [256, 256]         # override
window_size = 24                   # override
sigma = 1.005                      # override
length_scale = 7.0                 # override
beta = 0.2                         # override
num_epochs = 40                    # override
data_type = "physionet"            # override

#==============================
# Hardcoded variables
#==============================
data_dim = 35 # features in Physionet
time_length = 48 # length for a single patient (48 hours)

### 3.1 Reload a model checkpoint

In [114]:
import os
import pandas as pd
checkpoint_dir = 'external/models/251123_reproduce_physionet'

In [115]:
# results.iloc[:, 10:-8]
# results.kernel_scales[0]

In [116]:
# Create the model before restoring the checkpoint
from imputegap.wrapper.AlgoPython.GPVAE.models.models import *

# read the information contained in results.tsv to build the same model used 
# during training
results = pd.read_csv(os.path.join(checkpoint_dir, "results.tsv"), delimiter="\t")

# build the model
encoder = BandedJointEncoder
decoder = GaussianDecoder
image_preprocessor = None

data_type = results.data[0]
kernel = results.kernel[0]
beta = results.beta[0]
window_size = int(results.window_size[0])
kernel_scales = results.kernel_scales[0]
sigma = results.sigma[0]
length_scale = results.length_scale[0]
encoder_sizes = [results.encoder_width[0]]*results.encoder_depth[0]
decoder_sizes = [results.decoder_width[0]]*results.decoder_depth[0]

# In the results.tsv there are some missing information that we should 
# include (TODO)
# Currently we use the ones used for the physionet training
# preprocessor needed or not?
# (cnn_kernel_size, cnn_sizes)
# latent_dim
# data_dim
# time_length
# encoder
# decoder
# image_preprocessor
# M
# K

# instantiate the model
model = GP_VAE(
    latent_dim=latent_dim, 
    data_dim=data_dim, 
    time_length=time_length,
    encoder_sizes=encoder_sizes,
    encoder=encoder,
    decoder_sizes=decoder_sizes,
    decoder=decoder,
    kernel=kernel, 
    sigma=sigma,
    length_scale=length_scale, 
    kernel_scales = kernel_scales,
    image_preprocessor=image_preprocessor, 
    window_size=window_size,
    beta=beta, 
    M=M,
    K=K,
    data_type=data_type
)

In [117]:
# create a checkpoint with the built model
if image_preprocessor is None:
  checkpoint = tf.train.Checkpoint(
      encoder=model.encoder.net,
      decoder=model.decoder.net,
  )
else:
  checkpoint = tf.train.Checkpoint(
      encoder=model.encoder.net,
      decoder=model.decoder.net,
      preprocessor=model.preprocessor.net
  )

# restore the latest checkpoint in the checkpoint_dir
latest = tf.train.latest_checkpoint(checkpoint_dir)
checkpoint.restore(latest).expect_partial()
print("Checkpoint restored.")

Checkpoint restored.


### 3.2. Contaminate data and reconstruct it for an individual patient

In [118]:
from imputegap.recovery.imputation import Imputation
from imputegap.recovery.manager import TimeSeries
from imputegap.tools import utils

# initialize the time series object
ts = TimeSeries()

# load and normalize the dataset
ts.load_series("./external/datasets/test.txt")
ts.normalize(normalizer="z_score")

# patient with id 27
sample_idx = 27
values_one_sample = 48
# Idx to access the first value for a patient with a given sample idx (flattened dimension)
flatten_idx = sample_idx*values_one_sample

# Add missingness to the data, ts has shape (T, V)
ts_m = ts.Contamination.mcar(ts.data[:,flatten_idx:flatten_idx+48], rate_dataset=0.2, rate_series=0.4, block_size=10, seed=True)

ts.plot(ts.data[:,flatten_idx:flatten_idx+48], ts_m, subplot=True)

# ts_m contains NaN values, I should first create a mask, and then set NaN to 0.
ts_m_mask = np.isnan(ts_m)
ts_m_compatible = np.nan_to_num(ts_m, nan=0.0)


(SYS) The dataset is loaded from ./external/datasets/test.txt

> logs: normalization (z_score) of the data - runtime: 0.0660 seconds

(CONT) missigness pattern: MCAR
	selected series: 14, 16, 20, 22, 25, 27, 30
	percentage of contaminated series: 20.0%
	rate of missing data per series: 40.0%
	block size: 10
	security offset: [0-4]
	seed value: 42

plots saved in: ./imputegap_assets/25_11_29_23_30_11_imputegap_plot.jpg


In [119]:
# Pass to the encoder the contaminated sample
# the model takes (B, V, T), batches, values (time steps), time series (channels)
encoder_output = model.encode([ts_m_compatible.transpose()])

In [120]:
# Pass to the decoder the most probable latent trajectory (mean())
# and compute the most probable data space trajectory (mean())
decoder_output = model.decode(encoder_output.mean()).mean().numpy()

In [121]:
# set the original observed data back in the imputed data (in this way
# we actually impute the original data and not simply replace the entire data
# with what the model has returned)
# decoder output is (B, V, T)
decoder_output[0].transpose()[np.invert(ts_m_mask)] = ts_m[np.invert(ts_m_mask)]

In [122]:
# ts.data is (T, V), that's the reason we need to transpose
ts.plot(input_data=ts.data[:,flatten_idx:flatten_idx+48], incomp_data=ts_m, recov_data=decoder_output[0].transpose(), nbr_series=35, subplot=True, algorithm="GP-VAE", save_path="./imputegap_assets/imputation")


plots saved in: ./imputegap_assets/imputation/25_11_29_23_30_18_GP-VAE_plot.jpg


'./imputegap_assets/imputation/25_11_29_23_30_18_GP-VAE_plot.jpg'

In [74]:
import matplotlib.pyplot as plt
plt.plot(encoder_output.mean()[0, 34])
plt.show()


### Test code integration

In [123]:
from imputegap.recovery.manager import TimeSeries
from imputegap.tools import utils
from imputegap.algorithms.gp_vae import gp_vae

# initialize the time series object
ts = TimeSeries()

# load and normalize the dataset
ts.load_series("./external/datasets/test.txt")
ts.normalize(normalizer="z_score")

# patient with id 27
sample_idx = 27
values_one_sample = 48
# Idx to access the first value for a patient with a given sample idx (flattened dimension)
flatten_idx = sample_idx*values_one_sample

# Add missingness to the data, ts has shape (T, V)
ts_m = ts.Contamination.mcar(ts.data[:,flatten_idx:flatten_idx+48], rate_dataset=0.2, rate_series=0.4, block_size=10, seed=True)

ts.plot(ts.data[:,flatten_idx:flatten_idx+48], ts_m, subplot=True)

ts_m_imputed = gp_vae(ts_m, 'external/models/251123_reproduce_physionet')

ts.plot(input_data=ts.data[:,flatten_idx:flatten_idx+48], incomp_data=ts_m, recov_data=ts_m_imputed, nbr_series=35, subplot=True, algorithm="GP-VAE", save_path="./imputegap_assets/imputation")


(SYS) The dataset is loaded from ./external/datasets/test.txt

> logs: normalization (z_score) of the data - runtime: 0.0706 seconds

(CONT) missigness pattern: MCAR
	selected series: 14, 16, 20, 22, 25, 27, 30
	percentage of contaminated series: 20.0%
	rate of missing data per series: 40.0%
	block size: 10
	security offset: [0-4]
	seed value: 42

plots saved in: ./imputegap_assets/25_11_29_23_30_34_imputegap_plot.jpg
Path to a model checkpoint was given.
Checkpoint successfully restored.

plots saved in: ./imputegap_assets/imputation/25_11_29_23_30_40_GP-VAE_plot.jpg


'./imputegap_assets/imputation/25_11_29_23_30_40_GP-VAE_plot.jpg'

### Create a model from zero, aim of training it

In [ ]:
from imputegap.wrapper.AlgoPython.GPVAE.models.models import *
encoder = BandedJointEncoder
decoder = GaussianDecoder
image_preprocessor = None

model = GP_VAE(
    latent_dim=latent_dim, 
    data_dim=data_dim, 
    time_length=time_length,
    encoder_sizes=encoder_sizes,
    encoder=encoder,
    decoder_sizes=decoder_sizes,
    decoder=decoder,
    kernel=kernel, 
    sigma=sigma,
    length_scale=length_scale, 
    kernel_scales = kernel_scales,
    image_preprocessor=image_preprocessor, 
    window_size=window_size,
    beta=beta, 
    M=M,
    K=K,
    data_type=data_type
)

In [ ]:
dummy_x = tf.zeros([1, time_length, data_dim])
model(dummy_x)
model.summary()
enc = encoder(latent_dim, encoder_sizes)
out = enc(dummy_x)
